# Music Annotation Project V7.6: Correction-Safety Calibration

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook freezes the completed V7.5 `geometry_t80` candidate and asks one question:

> Which evidence available before clean-truth inspection predicts whether a proposed correction is safe enough for automatic execution?

## Experimental separation

- V7.5 reconstruction and routing are not changed or rerun.
- Only `geometry_t80` rows are used.
- Clean truth is used only to construct post-hoc outcome labels and damage measurements.
- Clean-derived fields are removed from predictors.
- Cross-validation is grouped by page, preventing components from the same page appearing in both training and test folds.
- Candidate rules are calibration findings only. This notebook does not authorise auto-acceptance.

## Primary endpoint

A component is labelled `safe` only if all configured outcome gates pass. Otherwise it is labelled `review_required`. The defaults are deliberately conservative and are recorded in the manifest. Sensitivity tables show how conclusions change across alternative label gates.

## Outputs

- `tables/calibration_dataset.csv`
- `tables/outcome_label_sensitivity.csv`
- `tables/univariate_predictor_screen.csv`
- `tables/page_grouped_cross_validation.csv`
- `tables/out_of_fold_predictions.csv`
- `tables/candidate_policy_thresholds.csv`
- `tables/calibration_confusion_by_page.csv`
- `charts/` and `contact_sheets/`
- `v7_6_manifest.json`


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas matplotlib scikit-learn


In [ ]:
import cv2, json, re, math, hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime, timezone
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix, brier_score_loss, balanced_accuracy_score
from sklearn.model_selection import GroupKFold, LeaveOneGroupOut
from sklearn.inspection import permutation_importance

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
RUN_OVERRIDE=None
OUTPUT_ROOT=ROOT/'correction_safety_calibration_v7_6'
MAX_CONTACTS=30
RANDOM_SEED=42

# Conservative outcome-label defaults. These are calibration definitions, not production thresholds.
LABEL_GATES={
    'maximum_component_mae':12.0,
    'minimum_ink_f1':0.95,
    'minimum_edge_f1':0.90,
    'maximum_removed_print_fraction':0.001,
    'maximum_preserved_annotation_fraction':0.02,
}

def complete_run(p):
    req=[p/'benchmark_manifest.json',p/'tables'/'surrounding_area_evidence_by_component.csv',p/'tables'/'surrounding_evidence_truth_diagnostics.csv',p/'tables'/'cache_status.csv']
    return all(x.exists() for x in req)
if RUN_OVERRIDE:
    V75_RUN=Path(RUN_OVERRIDE)
else:
    candidates=[p for p in (V75_ROOT/'runs').iterdir() if p.is_dir() and complete_run(p)] if (V75_ROOT/'runs').exists() else []
    if not candidates: raise RuntimeError('No complete V7.5 run found. Set RUN_OVERRIDE to the completed run folder.')
    V75_RUN=max(candidates,key=lambda p:p.stat().st_mtime)
RUN_NAME=datetime.now().strftime('%Y%m%d_%H%M%S')
OUT=OUTPUT_ROOT/RUN_NAME;TABLES=OUT/'tables';CHARTS=OUT/'charts';CONTACTS=OUT/'contact_sheets'
for d in [OUT,TABLES,CHARTS,CONTACTS]:d.mkdir(parents=True,exist_ok=True)
print('V7.5 source:',V75_RUN);print('V7.6 output:',OUT)


## Load V7.5 component evidence and truth diagnostics


In [ ]:
def read_required(path):
    if not path.exists(): raise RuntimeError(f'Missing required file: {path}')
    return pd.read_csv(path)
def normalise_method(df):
    for c in ['method','candidate','geometry_candidate']:
        if c in df.columns:
            return df[df[c].astype(str).str.lower().str.contains('t80')].copy()
    raise RuntimeError('No method/candidate field found to select geometry_t80.')
def key_columns(a,b):
    keys=[x for x in ['page','component','method','candidate','geometry_candidate'] if x in a.columns and x in b.columns]
    if not {'page','component'}.issubset(keys): raise RuntimeError('Evidence tables must share page and component.')
    return keys

evidence=normalise_method(read_required(V75_RUN/'tables'/'surrounding_area_evidence_by_component.csv'))
truth=normalise_method(read_required(V75_RUN/'tables'/'surrounding_evidence_truth_diagnostics.csv'))
keys=key_columns(evidence,truth)
truth_extra=[c for c in truth.columns if c not in keys]
df=evidence.merge(truth[keys+truth_extra],on=keys,how='inner',validate='one_to_one',suffixes=('','_truth'))
if df.empty: raise RuntimeError('No matched geometry_t80 evidence/truth rows.')
print('Matched geometry_t80 components:',len(df),'pages:',df.page.nunique())
print('Columns:',sorted(df.columns))


## Recover component-level reconstruction outcomes from persistent V7.5 caches


In [ ]:
SUPPORTED={'.png','.jpg','.jpeg','.tif','.tiff','.bmp'}
def image_map(folder): return {p.stem:p for p in Path(folder).iterdir() if p.is_file() and p.suffix.lower() in SUPPORTED} if Path(folder).exists() else {}
def printed_ink(im): return cv2.adaptiveThreshold(im,255,cv2.ADAPTIVE_THRESH_GAUSSIAN_C,cv2.THRESH_BINARY_INV,41,15)>0
def edges(im): return cv2.Canny(im,70,170)>0
def tolerant_f1(ref,pred,region,tol=1):
    ref=ref&region;pred=pred&region
    if not ref.sum(): return np.nan
    k=np.ones((2*tol+1,2*tol+1),np.uint8)
    precision=np.sum(pred&(cv2.dilate(ref.astype(np.uint8),k)>0))/max(pred.sum(),1)
    recall=np.sum(ref&(cv2.dilate(pred.astype(np.uint8),k)>0))/ref.sum()
    return float(2*precision*recall/max(precision+recall,1e-12))

clean_map=image_map(ROOT/'source'/'clean')
cache_status=read_required(V75_RUN/'tables'/'cache_status.csv')
cache_lookup={str(r.page):Path(r.cache) for _,r in cache_status.iterrows()}
rows=[]
for page,group in df.groupby('page'):
    page=str(page);cp=clean_map.get(page);zp=cache_lookup.get(page)
    if cp is None or zp is None or not cp.exists() or not zp.exists():
        print('Skipping unavailable page/cache:',page);continue
    clean=cv2.imread(str(cp),0);z=np.load(zp)
    if 'mask' not in z.files or 'geometry_t80' not in z.files: raise RuntimeError(f'{page}: cache lacks mask or geometry_t80')
    mask=z['mask']>0;recon=z['geometry_t80'];n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8)
    for _,r in group.iterrows():
        cid=int(r.component)
        if cid<=0 or cid>=n:continue
        comp=lab==cid;diff=np.abs(clean.astype(np.float32)-recon.astype(np.float32))
        rows.append({'page':page,'component':cid,'component_pixels':int(comp.sum()),'component_mae':float(diff[comp].mean()),'component_ink_f1':tolerant_f1(printed_ink(clean),printed_ink(recon),comp),'component_edge_f1':tolerant_f1(edges(clean),edges(recon),comp),'x_cache':int(stats[cid,0]),'y_cache':int(stats[cid,1]),'w_cache':int(stats[cid,2]),'h_cache':int(stats[cid,3])})
outcomes=pd.DataFrame(rows)
df=df.merge(outcomes,on=['page','component'],how='inner',validate='one_to_one')
print('Components with reconstructed outcomes:',len(df),'pages:',df.page.nunique())


## Construct clean-truth labels without exposing truth fields to predictors


In [ ]:
def first_existing(frame,names,default=np.nan):
    for n in names:
        if n in frame.columns:return pd.to_numeric(frame[n],errors='coerce')
    return pd.Series(default,index=frame.index,dtype=float)

removed_print=first_existing(df,['removed_clean_print_fraction','remove_clean_print_fraction','removed_print_fraction'],0.0)
preserved_annotation=1-first_existing(df,['preserved_clean_print_precision','preserve_clean_print_precision'],1.0)
ink_ok=df.component_ink_f1.fillna(1.0)>=LABEL_GATES['minimum_ink_f1']
edge_ok=df.component_edge_f1.fillna(1.0)>=LABEL_GATES['minimum_edge_f1']
mae_ok=df.component_mae<=LABEL_GATES['maximum_component_mae']
remove_ok=removed_print.fillna(0)<=LABEL_GATES['maximum_removed_print_fraction']
preserve_ok=preserved_annotation.fillna(0)<=LABEL_GATES['maximum_preserved_annotation_fraction']
df['safe']=mae_ok&ink_ok&edge_ok&remove_ok&preserve_ok
df['outcome']='safe';df.loc[~df.safe,'outcome']='review_required'
df['failure_reasons']=''
for cond,name in [(~mae_ok,'mae'),(~ink_ok,'ink'),(~edge_ok,'edge'),(~remove_ok,'removed_print'),(~preserve_ok,'preserved_annotation')]:
    df.loc[cond,'failure_reasons']=df.loc[cond,'failure_reasons'].map(lambda x:(x+'|'+name).strip('|'))

TRUTH_PATTERNS=['clean','truth','precision','paper','print','component_mae','component_ink','component_edge','safe','outcome','failure','x_cache','y_cache','w_cache','h_cache']
IDENTIFIERS=['page','component','method','candidate','geometry_candidate']
numeric=[c for c in df.select_dtypes(include=[np.number,bool]).columns if c not in IDENTIFIERS]
predictors=[c for c in numeric if not any(p in c.lower() for p in TRUTH_PATTERNS)]
# Remove invalid/sentinel and effectively constant fields.
X=df[predictors].apply(pd.to_numeric,errors='coerce').replace([np.inf,-np.inf],np.nan)
for c in list(X.columns):
    finite=X[c].dropna()
    if finite.empty or finite.nunique()<=1 or (finite.abs()>1e9).mean()>0.01:X=X.drop(columns=c)
predictors=list(X.columns)
if not predictors:raise RuntimeError('No eligible pre-truth predictors remain.')
print(df.outcome.value_counts());print('Predictors:',predictors)
df.to_csv(TABLES/'calibration_dataset.csv',index=False)


## Label-gate sensitivity


In [ ]:
sensitivity=[]
for mae_gate in [8,12,16,20]:
  for ink_gate in [.90,.95,.98]:
    for edge_gate in [.85,.90,.95]:
      label=(df.component_mae<=mae_gate)&(df.component_ink_f1.fillna(1)>=ink_gate)&(df.component_edge_f1.fillna(1)>=edge_gate)&(removed_print.fillna(0)<=LABEL_GATES['maximum_removed_print_fraction'])&(preserved_annotation.fillna(0)<=LABEL_GATES['maximum_preserved_annotation_fraction'])
      sensitivity.append({'mae_gate':mae_gate,'ink_f1_gate':ink_gate,'edge_f1_gate':edge_gate,'safe_components':int(label.sum()),'review_components':int((~label).sum()),'safe_fraction':float(label.mean())})
pd.DataFrame(sensitivity).to_csv(TABLES/'outcome_label_sensitivity.csv',index=False)


## Univariate evidence screen


In [ ]:
def auc_oriented(y,x):
    valid=pd.DataFrame({'y':y,'x':x}).dropna()
    if valid.y.nunique()<2 or valid.x.nunique()<2:return np.nan,np.nan
    auc=roc_auc_score(valid.y,valid.x);return float(max(auc,1-auc)),1 if auc>=.5 else -1
screen=[]
y=df.safe.astype(int)
for c in predictors:
    auc,direction=auc_oriented(y,X[c])
    safe=df.loc[df.safe,c].replace([np.inf,-np.inf],np.nan).dropna();review=df.loc[~df.safe,c].replace([np.inf,-np.inf],np.nan).dropna()
    screen.append({'predictor':c,'oriented_auc':auc,'safe_direction':direction,'safe_median':float(safe.median()) if len(safe) else np.nan,'review_median':float(review.median()) if len(review) else np.nan,'missing_fraction':float(X[c].isna().mean())})
screen_df=pd.DataFrame(screen).sort_values('oriented_auc',ascending=False);screen_df.to_csv(TABLES/'univariate_predictor_screen.csv',index=False);display(screen_df.head(25))


## Page-grouped out-of-fold calibration


In [ ]:
groups=df.page.astype(str);unique_pages=groups.nunique()
if unique_pages<3:raise RuntimeError('At least three pages are required for grouped calibration.')
splitter=LeaveOneGroupOut() if unique_pages<=12 else GroupKFold(n_splits=min(10,unique_pages))
models={
 'logistic':Pipeline([('impute',SimpleImputer(strategy='median')),('scale',StandardScaler()),('model',LogisticRegression(class_weight='balanced',max_iter=4000,random_state=RANDOM_SEED))]),
 'forest':Pipeline([('impute',SimpleImputer(strategy='median')),('model',RandomForestClassifier(n_estimators=500,min_samples_leaf=5,max_features='sqrt',class_weight='balanced_subsample',random_state=RANDOM_SEED,n_jobs=-1))])}
records=[];fold_rows=[]
for name,model in models.items():
    oof=np.full(len(df),np.nan);fold=0
    for train,test in splitter.split(X,y,groups):
        fold+=1
        if y.iloc[train].nunique()<2:continue
        model.fit(X.iloc[train],y.iloc[train]);oof[test]=model.predict_proba(X.iloc[test])[:,1]
        valid=np.isfinite(oof[test]);yt=y.iloc[test].to_numpy()[valid];pt=oof[test][valid]
        fold_rows.append({'model':name,'fold':fold,'test_pages':'|'.join(sorted(groups.iloc[test].unique())),'components':len(test),'safe_components':int(yt.sum()),'roc_auc':roc_auc_score(yt,pt) if len(np.unique(yt))==2 else np.nan,'average_precision':average_precision_score(yt,pt) if yt.sum()>0 else np.nan,'brier':brier_score_loss(yt,pt)})
    valid=np.isfinite(oof)
    if y[valid].nunique()<2:raise RuntimeError(f'{name}: grouped predictions lack both outcome classes.')
    records.append({'model':name,'roc_auc':roc_auc_score(y[valid],oof[valid]),'average_precision':average_precision_score(y[valid],oof[valid]),'brier':brier_score_loss(y[valid],oof[valid]),'components':int(valid.sum()),'pages':int(groups[valid].nunique())})
    df[f'prob_safe_{name}']=oof
cv_summary=pd.DataFrame(records).sort_values(['average_precision','roc_auc'],ascending=False);cv_summary.to_csv(TABLES/'page_grouped_cross_validation.csv',index=False);pd.DataFrame(fold_rows).to_csv(TABLES/'page_grouped_fold_results.csv',index=False);display(cv_summary)
best_model=cv_summary.iloc[0].model;prob=df[f'prob_safe_{best_model}'].to_numpy()


## Search conservative candidate safety policies


In [ ]:
# A candidate auto-safe threshold is selected by bounding false-safe outcomes, not by maximising accuracy.
policies=[]
for max_false_safe_rate in [0.00,0.005,0.01,0.02,0.05]:
    for threshold in np.linspace(0,1,1001):
        pred=prob>=threshold
        if not pred.any():continue
        false_safe=int(np.sum(pred&~df.safe.to_numpy()));selected=int(pred.sum());rate=false_safe/selected
        if rate<=max_false_safe_rate:
            policies.append({'model':best_model,'maximum_allowed_false_safe_rate':max_false_safe_rate,'probability_threshold':float(threshold),'selected_components':selected,'selected_fraction':selected/len(df),'false_safe_components':false_safe,'observed_false_safe_rate':rate,'safe_recall':float(np.sum(pred&df.safe.to_numpy())/max(df.safe.sum(),1))});break
policy_df=pd.DataFrame(policies);policy_df.to_csv(TABLES/'candidate_policy_thresholds.csv',index=False);display(policy_df)
# Use strictest non-empty policy for diagnostic confusion tables; still not a production threshold.
selected_policy=policy_df.iloc[0] if not policy_df.empty else None
if selected_policy is not None:
    t=float(selected_policy.probability_threshold);df['candidate_auto_safe']=prob>=t
else:
    t=1.0;df['candidate_auto_safe']=False
page_conf=df.groupby('page').apply(lambda g:pd.Series({'components':len(g),'true_safe':int(g.safe.sum()),'candidate_auto_safe':int(g.candidate_auto_safe.sum()),'false_safe':int((g.candidate_auto_safe&~g.safe).sum()),'missed_safe':int((~g.candidate_auto_safe&g.safe).sum())}),include_groups=False).reset_index()
page_conf.to_csv(TABLES/'calibration_confusion_by_page.csv',index=False)
df[['page','component','safe','outcome','failure_reasons',f'prob_safe_{best_model}','candidate_auto_safe']].to_csv(TABLES/'out_of_fold_predictions.csv',index=False)


## Fit final explanatory models and calculate feature importance


In [ ]:
importance=[]
for name,model in models.items():
    model.fit(X,y)
    p=permutation_importance(model,X,y,n_repeats=30,random_state=RANDOM_SEED,scoring='average_precision')
    for c,mean,std in zip(predictors,p.importances_mean,p.importances_std):importance.append({'model':name,'predictor':c,'importance_mean':float(mean),'importance_std':float(std)})
importance_df=pd.DataFrame(importance).sort_values(['model','importance_mean'],ascending=[True,False]);importance_df.to_csv(TABLES/'permutation_feature_importance.csv',index=False)
for name,g in importance_df.groupby('model'):
    q=g.head(20).sort_values('importance_mean');plt.figure(figsize=(9,7));plt.barh(q.predictor,q.importance_mean,xerr=q.importance_std);plt.title(f'V7.6 permutation importance: {name}');plt.xlabel('Decrease in average precision');plt.tight_layout();plt.savefig(CHARTS/f'feature_importance_{name}.png',dpi=170);plt.close()


## Calibration charts and diagnostic contact sheets


In [ ]:
plt.figure(figsize=(7,6))
for name in models:
    p=df[f'prob_safe_{name}'];bins=pd.qcut(p.rank(method='first'),q=min(10,len(p)),duplicates='drop');cal=df.assign(_p=p,_bin=bins).groupby('_bin',observed=True).agg(predicted=('_p','mean'),observed=('safe','mean'))
    plt.plot(cal.predicted,cal.observed,marker='o',label=name)
plt.plot([0,1],[0,1],'k--');plt.xlabel('Predicted safe probability');plt.ylabel('Observed safe fraction');plt.title('Page-grouped out-of-fold calibration');plt.legend();plt.tight_layout();plt.savefig(CHARTS/'calibration_curve.png',dpi=170);plt.close()

cache_status_map={str(r.page):Path(r.cache) for _,r in cache_status.iterrows()}
def contact(row,rank,kind):
    page=str(row.page);z=np.load(cache_status_map[page]);clean=cv2.imread(str(clean_map[page]),0);obs=z['observed'] if 'observed' in z.files else None;mask=z['mask']>0;recon=z['geometry_t80'];n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8);cid=int(row.component)
    x,y,w,h=stats[cid,:4];m=40;x0=max(0,x-m);y0=max(0,y-m);x1=min(clean.shape[1],x+w+m);y1=min(clean.shape[0],y+h+m)
    panels=[clean[y0:y1,x0:x1],recon[y0:y1,x0:x1],(lab[y0:y1,x0:x1]==cid).astype(np.uint8)*255];titles=['clean truth','frozen geometry_t80','component mask']
    if obs is not None:panels.insert(1,obs[y0:y1,x0:x1]);titles.insert(1,'observed')
    fig,axes=plt.subplots(1,len(panels),figsize=(5*len(panels),5));
    for ax,im,title in zip(np.atleast_1d(axes),panels,titles):ax.imshow(im,cmap='gray',vmin=0,vmax=255);ax.set_title(title);ax.axis('off')
    fig.suptitle(f'{kind} #{rank} | {page} component {cid} | true={row.outcome} | p_safe={row[f"prob_safe_{best_model}"]:.3f} | {row.failure_reasons}')
    plt.tight_layout();path=CONTACTS/f'{kind}_{rank:03d}_{page}_component_{cid:04d}.png';plt.savefig(path,dpi=170,bbox_inches='tight');plt.close(fig)

false_safe=df[df.candidate_auto_safe&~df.safe].sort_values(f'prob_safe_{best_model}',ascending=False).head(MAX_CONTACTS)
false_review=df[(~df.candidate_auto_safe)&df.safe].sort_values(f'prob_safe_{best_model}').head(MAX_CONTACTS)
for kind,data in [('false_safe',false_safe),('missed_safe',false_review)]:
    for rank,(_,row) in enumerate(data.iterrows(),1):contact(row,rank,kind)
print('Diagnostic contact sheets:',CONTACTS)


## Export manifest and executive findings


In [ ]:
summary={'created_utc':datetime.now(timezone.utc).isoformat(),'source_v7_5_run':str(V75_RUN),'frozen_candidate':'geometry_t80','clean_truth_role':'post-hoc labels and scoring only','page_grouped_cross_validation':True,'components':int(len(df)),'pages':int(df.page.nunique()),'safe_components':int(df.safe.sum()),'review_required_components':int((~df.safe).sum()),'label_gates':LABEL_GATES,'predictors':predictors,'excluded_predictor_patterns':TRUTH_PATTERNS,'best_cross_validated_model':best_model,'cross_validation':cv_summary.to_dict('records'),'candidate_policy_is_production_authorisation':False,'selected_diagnostic_threshold':t,'outputs':{'tables':str(TABLES),'charts':str(CHARTS),'contact_sheets':str(CONTACTS)}}
(OUT/'v7_6_manifest.json').write_text(json.dumps(summary,indent=2))
lines=['V7.6 CORRECTION-SAFETY CALIBRATION','',f'Source V7.5 run: {V75_RUN}','Frozen candidate: geometry_t80',f'Components: {len(df)} across {df.page.nunique()} pages',f'Safe under primary label: {int(df.safe.sum())}',f'Review required: {int((~df.safe).sum())}','',f'Best page-grouped model: {best_model}','', 'IMPORTANT: candidate thresholds are retrospective calibration findings, not permission for auto-acceptance. Independent validation remains required.']
(OUT/'executive_summary.txt').write_text('\n'.join(lines))
print('\n'.join(lines));print('V7.6 COMPLETE:',OUT)
